<a href="https://colab.research.google.com/github/Otza02/satellite-img-segmentation/blob/main/notebooks/cross-val.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Otza02/satellite-img-segmentation.git
%cd satellite-img-segmentation
%pip install -e .

Cloning into 'satellite-img-segmentation'...
remote: Enumerating objects: 325, done.
remote: Counting objects: 100% (325/325), done.
remote: Compressing objects: 100% (169/169), done.
remote: Total 325 (delta 165), reused 254 (delta 109), pack-reused 0 (from 0)
Receiving objects: 100% (325/325), 1.61 MiB | 42.30 MiB/s, done.
Resolving deltas: 100% (165/165), done.
/content/satellite-img-segmentation
Obtaining file:///content/satellite-img-segmentation
  Preparing metadata (setup.py) ... done
  Running setup.py develop for satelliteSegmentation


## Reiniciar sesion

In [1]:
%cd /content/satellite-img-segmentation
!unzip -q data/train.zip -d data/train

/content/satellite-img-segmentation


## Cross Validation

In [10]:
from satelliteSegmentation.dataset import AugmentedSubset, SatelliteData, kfold_loaders
from satelliteSegmentation.train import train_model
from satelliteSegmentation.config import Config
from satelliteSegmentation.metrics import segmentation_metrics
from satelliteSegmentation.models import UNet

from torchvision.transforms import v2

import torch
from torch.utils.data import DataLoader

import json
from pathlib import Path

In [11]:
def calc_weights(
    dataloader,
    num_classes: int,
    ignore_index: int,
):
    counts = torch.zeros(num_classes, dtype=torch.long)

    for _, masks in dataloader:
        masks = masks.flatten()
        masks = masks[masks != ignore_index]

        counts += torch.bincount(
            masks,
            minlength=num_classes,
        )

    # Solo clases que participan en el entrenamiento
    valid_counts = counts.clone()
    valid_counts[ignore_index] = 0

    # Ejemplo: inverse square root frequency
    weights = torch.zeros(num_classes, dtype=torch.float32)

    valid = valid_counts > 0
    weights[valid] = 1.0 / torch.sqrt(valid_counts[valid].float())

    # Normalizar para que la media de los weights válidos sea 1
    weights[valid] /= weights[valid].mean()

    # El weight de la clase ignorada es irrelevante
    weights[ignore_index] = 0.0

    return counts, weights

# counts, weights = calc_weights(
#     train_loader,
#     num_classes=7,
#     ignore_index=6,
# )
# weights = weights.to(device)

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 64

In [13]:
data = SatelliteData("data/train")
shape_tf = v2.Compose([
    v2.RandomHorizontalFlip(),
    v2.RandomVerticalFlip(),
    v2.RandomRotation([0, 180], fill=6),
])

color_tf = v2.Compose([
    v2.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.1)
])

100%|██████████| 5184/5184 [00:08<00:00, 616.59it/s]


Dataset cargado:
X shape = torch.Size([5184, 3, 120, 120])
Y shape = torch.Size([5184, 120, 120])


In [14]:
loader_kwargs = {
    "num_workers": 2,
    "pin_memory": device == "cuda",
    "persistent_workers": device == "cuda",
}

k = 5
results = {}
for fold, train_loader, val_loader in kfold_loaders(
    data,
    k=k,
    batch_size=batch_size,
    shape_transform=shape_tf,
    color_transform=color_tf,
    **loader_kwargs,
):
    _, weights = calc_weights(
        train_loader,
        num_classes=7,
        ignore_index=6,
    )
    weights = weights.to(device)
    config = Config(device, weight_decay=2e-5, kernel_size=3, lr=1.51e-04, weights=weights, patience=15)
    print(f"=== Fold {fold + 1}/{k} ===")
    model = UNet(config)
    criterion = torch.nn.CrossEntropyLoss(config.weights, ignore_index=6)
    model, hist = train_model(model, train_loader, val_loader, criterion, config)
    metrics = segmentation_metrics(model, val_loader, config.device, config.num_classes)

    results[f"fold={fold}"] = {
        "hist": hist,
        "metrics": metrics,
    }

=== Fold 1/5 ===
Epoch 01/100 | train_loss=1.1430 | val_loss=1.0294 | time 00:16
Epoch 02/100 | train_loss=0.8269 | val_loss=0.8981 | time 00:16
Epoch 03/100 | train_loss=0.7029 | val_loss=0.9077 | time 00:15
Epoch 04/100 | train_loss=0.6507 | val_loss=0.7498 | time 00:15
Epoch 05/100 | train_loss=0.6050 | val_loss=0.7640 | time 00:15
Epoch 06/100 | train_loss=0.5704 | val_loss=0.6738 | time 00:15
Epoch 07/100 | train_loss=0.5435 | val_loss=0.7677 | time 00:15
Epoch 08/100 | train_loss=0.5196 | val_loss=0.7046 | time 00:15
Epoch 09/100 | train_loss=0.5068 | val_loss=0.6260 | time 00:15
Epoch 10/100 | train_loss=0.4885 | val_loss=0.6374 | time 00:15
Epoch 11/100 | train_loss=0.4737 | val_loss=0.6514 | time 00:15
Epoch 12/100 | train_loss=0.4734 | val_loss=0.6170 | time 00:15
Epoch 13/100 | train_loss=0.4461 | val_loss=0.6411 | time 00:15
Epoch 14/100 | train_loss=0.4575 | val_loss=0.6534 | time 00:15
Epoch 15/100 | train_loss=0.4425 | val_loss=0.6191 | time 00:15
Epoch 16/100 | train_lo

In [25]:
from copy import deepcopy

results_backup = deepcopy(results)

In [26]:
results[f"fold=0"]["metrics"]["confusion_matrix"].tolist()

for fold in results.keys():
    results[fold]["metrics"]["confusion_matrix"] = results[fold]["metrics"]["confusion_matrix"].tolist()

In [28]:
result_folder = Path("result/cross-val")
result_folder.mkdir(parents=True, exist_ok=True)
with open(result_folder / "models.json", "w") as f:
    json.dump(results, f)